# Monte Carlo Stochastic Blend

Uses pre-saved OOF predictions (no retraining). Samples 10k random Dirichlet weight
vectors, evaluates each on OOF AUC, submits the best-found blend for each model set.

**Note**: Previously run in s6e2-ensemble.ipynb with identical OOF files.  
This re-run uses the saved .npy files and also submits the MC-weighted test preds  
(previously only SLSQP equal-weight was submitted).

**Baseline to beat**: catboost_default_proba LB 0.95356

In [1]:
import subprocess
import random
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize

random.seed(42)
np.random.seed(42)
N_SAMPLES = 10_000

KAGGLE_DATA = Path('/kaggle/input/playground-series-s6e2')
LOCAL_DATA  = Path('data')
DATA_DIR    = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(pd.read_csv(DATA_DIR / 'train.csv'))
test  = prep(pd.read_csv(DATA_DIR / 'test.csv'))
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')
y     = train['heart_disease'].values

## Load Saved OOF & Test Predictions

In [2]:
oof_cat  = np.load('submissions/oof_cat.npy')
oof_lgb  = np.load('submissions/oof_lgb.npy')
oof_xgb  = np.load('submissions/oof_xgb.npy')
test_cat = np.load('submissions/test_cat.npy')
test_lgb = np.load('submissions/test_lgb.npy')
test_xgb = np.load('submissions/test_xgb.npy')

print('Individual OOF AUCs:')
for name, oof in [('CatBoost', oof_cat), ('LGBM', oof_lgb), ('XGBoost', oof_xgb)]:
    print(f'  {name:<10} {roc_auc_score(y, oof):.5f}')

Individual OOF AUCs:
  CatBoost   0.95532
  LGBM       0.95523
  XGBoost    0.95525


## Generate LR OOF
LR needs retraining (no saved OOF). Fast on CPU — ~2 min for 5 folds.

In [3]:
CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
FEATURES     = NUM_FEATURES + CAT_FEATURES

X_lr      = pd.get_dummies(train[FEATURES], columns=CAT_FEATURES)
X_test_lr = pd.get_dummies(test[FEATURES],  columns=CAT_FEATURES)
X_test_lr = X_test_lr.reindex(columns=X_lr.columns, fill_value=0)

scaler = StandardScaler()
X_lr_s      = scaler.fit_transform(X_lr)
X_test_lr_s = scaler.transform(X_test_lr)

skf       = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_lr    = np.zeros(len(y))
test_lr_folds = np.zeros((5, len(test)))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_lr_s, y)):
    lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42, n_jobs=-1)
    lr.fit(X_lr_s[tr_idx], y[tr_idx])
    oof_lr[val_idx]       = lr.predict_proba(X_lr_s[val_idx])[:, 1]
    test_lr_folds[fold]   = lr.predict_proba(X_test_lr_s)[:, 1]
    print(f'  LR Fold {fold+1}  AUC={roc_auc_score(y[val_idx], oof_lr[val_idx]):.5f}')

test_lr = test_lr_folds.mean(axis=0)
print(f'  LR OOF AUC: {roc_auc_score(y, oof_lr):.5f}')

np.save('submissions/oof_lr.npy',  oof_lr)
np.save('submissions/test_lr.npy', test_lr)

  LR Fold 1  AUC=0.95324
  LR Fold 2  AUC=0.95226
  LR Fold 3  AUC=0.95297
  LR Fold 4  AUC=0.95259
  LR Fold 5  AUC=0.95332
  LR OOF AUC: 0.95288


## Monte Carlo Blend — 3 GBTs

In [4]:
# Sample N_SAMPLES random weight vectors on the simplex (Dirichlet alpha=1 = uniform)
weights3 = np.random.dirichlet(alpha=[1, 1, 1], size=N_SAMPLES)
oofs3    = np.stack([oof_cat, oof_lgb, oof_xgb], axis=1)  # (N, 3)
scores3  = [roc_auc_score(y, weights3[i] @ oofs3.T) for i in range(N_SAMPLES)]

best_idx3 = np.argmax(scores3)
best_w3   = weights3[best_idx3]
best_s3   = scores3[best_idx3]

simple_avg3 = roc_auc_score(y, oofs3.mean(axis=1))

print(f'Simple avg (3 GBTs):  {simple_avg3:.5f}')
print(f'MC best   (3 GBTs):   {best_s3:.5f}')
print(f'  cat={best_w3[0]:.3f}  lgb={best_w3[1]:.3f}  xgb={best_w3[2]:.3f}')

Simple avg (3 GBTs):  0.95537
MC best   (3 GBTs):   0.95538
  cat=0.546  lgb=0.208  xgb=0.246


## Monte Carlo Blend — 3 GBTs + LR

In [5]:
weights4 = np.random.dirichlet(alpha=[1, 1, 1, 1], size=N_SAMPLES)
oofs4    = np.stack([oof_cat, oof_lgb, oof_xgb, oof_lr], axis=1)
scores4  = [roc_auc_score(y, weights4[i] @ oofs4.T) for i in range(N_SAMPLES)]

best_idx4 = np.argmax(scores4)
best_w4   = weights4[best_idx4]
best_s4   = scores4[best_idx4]

simple_avg4 = roc_auc_score(y, oofs4.mean(axis=1))

print(f'Simple avg (3 GBTs + LR): {simple_avg4:.5f}')
print(f'MC best   (3 GBTs + LR):  {best_s4:.5f}')
print(f'  cat={best_w4[0]:.3f}  lgb={best_w4[1]:.3f}  xgb={best_w4[2]:.3f}  lr={best_w4[3]:.3f}')

Simple avg (3 GBTs + LR): 0.95516
MC best   (3 GBTs + LR):  0.95538
  cat=0.532  lgb=0.202  xgb=0.265  lr=0.001


## Summary & Submit All

In [6]:
tests3 = np.stack([test_cat, test_lgb, test_xgb], axis=1)
tests4 = np.stack([test_cat, test_lgb, test_xgb, test_lr], axis=1)

ON_KAGGLE = Path('/kaggle/working').exists()

to_submit = [
    ('mc_blend_3gbt',     best_w3, tests3, best_s3),
    ('mc_blend_3gbt_lr',  best_w4, tests4, best_s4),
    ('simple_avg_3gbt',   np.array([1/3, 1/3, 1/3]),          tests3, simple_avg3),
    ('simple_avg_3gbt_lr',np.array([1/4, 1/4, 1/4, 1/4]),     tests4, simple_avg4),
]

print(f'=== Summary ===')
for label, w, _, score in to_submit:
    print(f'  {label:<25} OOF={score:.5f}  weights={[round(x,3) for x in w]}')

print()
for label, w, tests, cv_auc in to_submit:
    pred  = (w @ tests.T)
    fname = f'submissions/{label}.csv'
    sub   = ss.copy()
    sub['Heart Disease'] = pred
    sub.to_csv(fname, index=False)
    desc = f'{label} | cv_auc={cv_auc:.4f}'
    if ON_KAGGLE:
        print(f'Submit: kaggle competitions submit -c playground-series-s6e2 -f {fname} -m "{desc}"')
    else:
        r = subprocess.run(
            ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
             '-f', fname, '-m', desc],
            capture_output=True, text=True
        )
        status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:80]
        print(f'{label}: {status}')

=== Summary ===
  mc_blend_3gbt             OOF=0.95538  weights=[0.546, 0.208, 0.246]
  mc_blend_3gbt_lr          OOF=0.95538  weights=[0.532, 0.202, 0.265, 0.001]
  simple_avg_3gbt           OOF=0.95537  weights=[0.333, 0.333, 0.333]
  simple_avg_3gbt_lr        OOF=0.95516  weights=[0.25, 0.25, 0.25, 0.25]

mc_blend_3gbt: submitted
mc_blend_3gbt_lr: submitted
simple_avg_3gbt: submitted
simple_avg_3gbt_lr: submitted
